# HopLab Server (Colab)

**Uso:** `Runtime → Run all` (Ctrl+F9)

1. Ejecutá el setup (secciones 0–3).
2. Al final verás la **URL del túnel** — pegala en Vercel → **Conectar motor**.
3. Dejá corriendo la celda **Terminal** (sección 5) para ver el feedback del API en vivo.

> Runtime con **GPU**: `Runtime → Change runtime type → T4 GPU`


## 0. Configuración (editar)

Única celda que debes editar: repo, Drive e ID compartido.


In [ ]:
# ─── CONFIGURACIÓN (única que debes editar) ──────────────────────────────────
# Repo del engine (cámbialo cuando lo subas a GitHub)
REPO_URL    = "https://github.com/kaponeminusone/athlete-analysis"
REPO_BRANCH = "main"

# Google Drive: True = persistencia entre sesiones (recomendado)
USE_DRIVE   = True
DRIVE_DATA  = "/content/drive/MyDrive/hoplab-data"  # owner: carpeta propia en MyDrive

# Owner: deja vacío → MyDrive/hoplab-data (mkdir local basta).
#   Si pegas tu propio folder ID (p.ej. para probar la ruta compartida), las
#   subcarpetas se crean vía Drive API si FUSE mkdir falla (errno 95).
# Invitado: pega el ID de la carpeta compartida `hoplab-data` del OWNER.
# ID = segmento de la URL: https://drive.google.com/drive/folders/FOLDER_ID
# También puedes exportar HOPLAB_DATA_FOLDER_ID en el entorno antes de Run All.
HOPLAB_DATA_FOLDER_ID = ""

# Raíz alternativa (invitado): más fiable para escrituras que .shortcut-targets-by-id.
# Vacío = auto (ver orden en celda de Drive). Tras "Añadir acceso directo a Mi unidad"
# en la carpeta compartida, suele funcionar mejor:
#   HOPLAB_DATA_ROOT = "/content/drive/MyDrive/hoplab-data"
HOPLAB_DATA_ROOT = ""

# Puerto de la API
API_PORT    = 8000
# ─────────────────────────────────────────────────────────────────────────────


## 1. Solo owner (una vez)

Ejecuta **una sola vez** (antes de invitar invitados). Crea `MyDrive/hoplab-data` + subcarpetas e imprime el **folder ID** para `HOPLAB_DATA_FOLDER_ID`.

Los invitados **no** deben ejecutar esta celda. En runs posteriores del owner puedes omitirla (colapsá esta sección).


In [ ]:
# ─── SOLO OWNER: crear estructura Drive (ejecutar UNA vez; omitir después) ───
# Los invitados NO deben ejecutar esta celda.
# Copia equivalente: hoplab-cloud/colab/owner_bootstrap_drive.py

from __future__ import annotations

import pathlib

from google.colab import auth, drive
from googleapiclient.discovery import build

FOLDER_NAME = "hoplab-data"
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive") / FOLDER_NAME
SUBDIRS = ("videos", "output", "venues", "models", "logs")
_FOLDER_MIME = "application/vnd.google-apps.folder"


def _ensure_local_tree() -> None:
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    for name in SUBDIRS:
        (DRIVE_ROOT / name).mkdir(parents=True, exist_ok=True)
    print(f"✅ Estructura local lista: {DRIVE_ROOT}")
    for name in SUBDIRS:
        print(f"   · {name}/")


def _find_or_create_folder_id(service) -> str:
    """Busca `hoplab-data` bajo My Drive (root); la crea si no existe."""
    query = (
        f"name = '{FOLDER_NAME}' and mimeType = '{_FOLDER_MIME}' "
        "and 'root' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    if files:
        return files[0]["id"]

    meta = {
        "name": FOLDER_NAME,
        "mimeType": _FOLDER_MIME,
        "parents": ["root"],
    }
    created = service.files().create(body=meta, fields="id").execute()
    print(f"📁 Carpeta '{FOLDER_NAME}' creada en My Drive vía API.")
    return created["id"]


def _find_child_folder(service, parent_id: str, name: str):
    query = (
        f"name = '{name}' and mimeType = '{_FOLDER_MIME}' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def _ensure_subdirs_via_api(service, parent_id: str) -> None:
    for name in SUBDIRS:
        existing = _find_child_folder(service, parent_id, name)
        if existing:
            print(f"   · API: '{name}/' ya existe (id={existing})")
            continue
        meta = {
            "name": name,
            "mimeType": _FOLDER_MIME,
            "parents": [parent_id],
        }
        created = service.files().create(body=meta, fields="id").execute()
        print(f"   · API: creada '{name}/' (id={created['id']})")


print("🔐 Montando Google Drive (MyDrive)…")
drive.mount("/content/drive")

_ensure_local_tree()

print("🔑 Autenticando para Drive API v3…")
auth.authenticate_user()
service = build("drive", "v3")

folder_id = _find_or_create_folder_id(service)
print(f"📁 Asegurando subcarpetas vía API bajo {folder_id}…")
_ensure_subdirs_via_api(service, folder_id)
share_url = f"https://drive.google.com/drive/folders/{folder_id}"

print()
print("─── RESULTADO (owner) ───────────────────────────────────────────")
print(f"Ruta:       {DRIVE_ROOT}")
print(f"Folder ID:  {folder_id}")
print(f"URL share:  {share_url}")
print()
print("Comparte esa carpeta con los invitados (Editor) y dales:")
print(f'HOPLAB_DATA_FOLDER_ID = "{folder_id}"')
print("─────────────────────────────────────────────────────────────────")


## 2. Setup — GPU / Drive / Clone / Install / Env

Bloque de preparación. Colapsá esta sección tras el primer Run All si no necesitás re-ejecutarla.


### 2.1 GPU


In [ ]:
# ─── Verificar GPU ───────────────────────────────────────────────────────────
import subprocess, sys
try:
    result = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total",
                             "--format=csv,noheader"], capture_output=True, text=True)
    if result.returncode == 0:
        print("✅ GPU detectada:", result.stdout.strip())
    else:
        print("⚠️  Sin GPU — el análisis será más lento (CPU).")
        print("   Ir a: Runtime → Change runtime type → T4 GPU")
except FileNotFoundError:
    print("⚠️  nvidia-smi no disponible — sin GPU.")

# Verificar PyTorch (ya instalado en Colab)
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"PyTorch {torch.__version__} | CUDA disponible: {cuda_ok}")
except ImportError:
    print("PyTorch no encontrado — se instalará en la siguiente celda.")


### 2.2 Drive — montar y estructura


In [ ]:
# ─── Montar Drive y crear estructura de carpetas ─────────────────────────────
import errno
import os
import pathlib
import time

_REQUIRED_SUBDIRS = ("videos", "output", "venues", "models", "logs")
_FOLDER_MIME = "application/vnd.google-apps.folder"

_ERR_MKDIR_SHARED = (
    "Google Drive FUSE no permite mkdir en esta ruta (errno 95 / EOPNOTSUPP), "
    "típico de carpetas vía `.shortcut-targets-by-id`.\n\n"
    "Con HOPLAB_DATA_FOLDER_ID definido, esta celda crea las subcarpetas vía "
    "Drive API (requiere auth.authenticate_user y rol Editor).\n"
    "Si aún falla: Organizar → «Añadir acceso directo a Mi unidad» y pon:\n"
    '  HOPLAB_DATA_ROOT = "/content/drive/MyDrive/hoplab-data"'
)


def ensure_dir(path: pathlib.Path) -> bool:
    """Intenta mkdir local. True si el dir existe; False si errno 95 y aún falta."""
    path = pathlib.Path(path)
    try:
        path.mkdir(parents=True, exist_ok=True)
        return True
    except OSError as e:
        if e.errno not in (errno.EOPNOTSUPP, 95):
            raise
        if path.exists() and path.is_dir():
            return True
        return False


def _drive_service():
    from google.colab import auth
    from googleapiclient.discovery import build
    print("🔑 Autenticando Drive API v3 (puede pedir consent)…")
    auth.authenticate_user()
    return build("drive", "v3")


def _find_child_folder(service, parent_id: str, name: str):
    query = (
        f"name = '{name}' and mimeType = '{_FOLDER_MIME}' "
        f"and '{parent_id}' in parents and trashed = false"
    )
    resp = (
        service.files()
        .list(q=query, spaces="drive", fields="files(id, name)", pageSize=10)
        .execute()
    )
    files = resp.get("files", [])
    return files[0]["id"] if files else None


def _ensure_subdir_via_api(service, parent_id: str, name: str) -> str:
    existing = _find_child_folder(service, parent_id, name)
    if existing:
        print(f"   · API: '{name}/' ya existe (id={existing})")
        return existing
    meta = {
        "name": name,
        "mimeType": _FOLDER_MIME,
        "parents": [parent_id],
    }
    created = service.files().create(body=meta, fields="id").execute()
    print(f"   · API: creada '{name}/' (id={created['id']})")
    return created["id"]


def _wait_fuse_subdir(root: pathlib.Path, name: str, timeout_s: float = 15.0) -> bool:
    target = root / name
    deadline = time.time() + timeout_s
    while time.time() < deadline:
        if target.is_dir():
            return True
        time.sleep(1.0)
    return target.is_dir()


def ensure_required_subdirs(data_root: pathlib.Path) -> None:
    """mkdir local primero; si FUSE falla y hay folder ID → Drive API v3."""
    folder_id = (HOPLAB_DATA_FOLDER_ID or os.environ.get("HOPLAB_DATA_FOLDER_ID", "")).strip()
    missing = []
    for sub in _REQUIRED_SUBDIRS:
        if not ensure_dir(data_root / sub):
            missing.append(sub)

    if not missing:
        return

    if not folder_id:
        raise OSError(
            errno.EOPNOTSUPP,
            f"{_ERR_MKDIR_SHARED}\n\n"
            f"Carpetas que faltan bajo {data_root}: {', '.join(missing)}\n"
            "(Sin HOPLAB_DATA_FOLDER_ID no hay fallback API; owner deja ID vacío "
            "y usa MyDrive, o define el folder ID.)",
        )

    print(
        f"⚠️  mkdir FUSE falló para: {', '.join(missing)}\n"
        f"   Creando vía Drive API bajo parent {folder_id}…"
    )
    service = _drive_service()
    for sub in missing:
        _ensure_subdir_via_api(service, folder_id, sub)

    still_missing = []
    for sub in missing:
        if _wait_fuse_subdir(data_root, sub):
            print(f"   ✓ FUSE ve {data_root / sub}")
        else:
            still_missing.append(sub)

    if still_missing:
        print(
            "ℹ️  Creadas en Drive API, pero FUSE aún no las muestra:\n"
            f"   {', '.join(still_missing)} bajo {data_root}\n"
            "   Re-ejecuta esta celda en unos segundos (lag de montaje)."
        )


def _resolve_data_root() -> pathlib.Path:
    """Orden: HOPLAB_DATA_ROOT → folder ID → MyDrive/hoplab-data."""
    explicit = (HOPLAB_DATA_ROOT or os.environ.get("HOPLAB_DATA_ROOT", "")).strip()
    if explicit:
        root = pathlib.Path(explicit)
        if not root.exists():
            raise FileNotFoundError(
                f"HOPLAB_DATA_ROOT no existe: {root}\n"
                "Comprueba el acceso directo en Mi unidad o corrige la ruta."
            )
        print(f"📂 Raíz explícita (HOPLAB_DATA_ROOT). DATA_ROOT = {root}")
        return root

    folder_id = (HOPLAB_DATA_FOLDER_ID or os.environ.get("HOPLAB_DATA_FOLDER_ID", "")).strip()
    if folder_id:
        shared = pathlib.Path("/content/drive/.shortcut-targets-by-id") / folder_id
        if shared.exists():
            print(
                f"📂 Carpeta compartida por ID. DATA_ROOT = {shared}\n"
                "   Nota: si mkdir FUSE falla (errno 95), se crean subcarpetas "
                "vía Drive API automáticamente."
            )
            return shared
        fallback = pathlib.Path(DRIVE_DATA)
        print(
            f"⚠️  No se encontró la carpeta compartida:\n"
            f"   {shared}\n"
            f"   Comprueba: (1) aceptaste la invitación de Drive del OWNER,\n"
            f"   (2) el ID es el de la carpeta `hoplab-data` (no un padre),\n"
            f"   (3) espera unos segundos tras montar Drive y vuelve a ejecutar esta celda.\n"
            f"   Intentando fallback: {fallback}"
        )
        if fallback.exists():
            print(f"📂 Fallback MyDrive. DATA_ROOT = {fallback}")
            return fallback
        raise FileNotFoundError(
            f"No existe la carpeta compartida ({shared}) ni el fallback ({fallback}). "
            "Acepta el share, verifica HOPLAB_DATA_FOLDER_ID y re-ejecuta tras montar Drive."
        )
    root = pathlib.Path(DRIVE_DATA)
    print(f"📂 Drive montado (MyDrive). DATA_ROOT = {root}")
    return root


if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_ROOT = _resolve_data_root()
else:
    DATA_ROOT = pathlib.Path("/content/hoplab-data")
    print(f"📂 Usando almacenamiento efímero. DATA_ROOT = {DATA_ROOT}")

ensure_required_subdirs(DATA_ROOT)

print("✅ Estructura de carpetas lista:")
for sub in ["videos", "output", "venues", "models"]:
    p = DATA_ROOT / sub
    n = sum(1 for _ in p.iterdir()) if p.exists() else 0
    print(f"   {p}  ({n} elementos)")


### 2.3 Clonar / actualizar engine


In [ ]:
# ─── Clonar / actualizar engine (efímero en /content) ────────────────────────
import subprocess, pathlib

ENGINE_DIR = pathlib.Path("/content/hoplab-engine")

if ENGINE_DIR.exists():
    print("Actualizando engine existente...")
    r = subprocess.run(["git", "-C", str(ENGINE_DIR), "pull", "--ff-only"],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
else:
    print(f"Clonando {REPO_URL} (rama {REPO_BRANCH})...")
    r = subprocess.run(["git", "clone", "--branch", REPO_BRANCH,
                        "--depth", "1", REPO_URL, str(ENGINE_DIR)],
                       capture_output=True, text=True)
    print(r.stdout or r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f"Error clonando repo: {r.stderr}")

print(f"✅ Engine listo en {ENGINE_DIR}")


### 2.4 Instalar dependencias


In [ ]:
# ─── Instalar dependencias ───────────────────────────────────────────────────
# PyTorch ya viene en Colab con CUDA. Solo instalamos el resto.
import subprocess, sys

REQS = ENGINE_DIR / "hoplab-cloud" / "colab" / "requirements-colab.txt"
if not REQS.exists():
    # Fallback: instalar directamente
    pkgs = ["ultralytics>=8.3", "opencv-python-headless>=4.9",
            "numpy>=1.26", "scipy>=1.13", "matplotlib>=3.9",
            "fastapi>=0.115", "uvicorn>=0.30", "python-multipart>=0.0.9"]
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + pkgs,
                       capture_output=True, text=True)
else:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                        "-r", str(REQS)], capture_output=True, text=True)

if r.returncode != 0:
    print("[ERROR pip]:", r.stderr[-2000:])
else:
    print("✅ Dependencias instaladas")

# Cachear pesos YOLO en Drive/models para no descargar cada sesión
import os
os.environ["YOLO_CONFIG_DIR"] = str(DATA_ROOT / "models")
print(f"   Pesos YOLO se cachearán en: {DATA_ROOT / 'models'}")


### 2.5 Env + symlinks

Caches en memoria (`TJ_FRAME_CACHE` / `TJ_ANNOTATED_CACHE`) + `TJ_PERSIST_FRAMES=1` y `TJ_WRITE_ANNOTATED=1` para que el análisis escriba `output/<video>/frames/*.jpg` **y** `annotated/*.jpg` en Drive. Así `/frame` (crudo y `annotated=1`) puede usar `FileResponse` en lugar de seek OpenCV sobre FUSE o re-anotar en cada request.

> **Tradeoff (`TJ_WRITE_ANNOTATED=1`):** persistir los frames anotados usa más espacio/cuota de escritura en Drive (~750 KB por frame) y añade algo de tiempo durante analyze/refine, pero elimina el costo de re-anotar en cada `/frame?annotated=1` → scrubbing fluido. Es la opción preferida cuando se prioriza la revisión ágil sobre el uso de disco. Si el espacio en Drive fuera crítico, poné `TJ_WRITE_ANNOTATED=0` y dejá `TJ_ANNOTATED_CACHE=1` (LRU en memoria): más lento tras cache-miss / reinicios, pero sin escribir a disco.

> **Thumbnails (`?w=`):** la UI puede pedir `/frame/<video>/<idx>?w=240` para recibir un JPEG reducido (~20–40 KB en vez de ~580–750 KB). Ideal para el scrub sobre el túnel; los bytes reducidos se cachean en el LRU existente.

> **Nota:** los primeros frames pueden seguir lentos hasta que caliente el LRU / existan JPEG en disco; al hacer scrub debería ir más rápido (cache hit + prefetch en la UI).


In [ ]:
# ─── Configurar env + symlinks Drive ←→ Engine ───────────────────────────────
import os, pathlib, sys

# Vars de entorno que api_server.py lee
os.environ["HOPLAB_OUTPUT_ROOT"] = str(DATA_ROOT / "output")
os.environ["HOPLAB_VENUE_ROOT"]  = str(DATA_ROOT / "venues")
os.environ["HOPLAB_VIDEO_ROOT"]  = str(DATA_ROOT / "videos")
os.environ["YOLO_CONFIG_DIR"]    = str(DATA_ROOT / "models")

# Optimización Drive: persistir annotated/*.jpg y frames crudos en disco.
# Así /frame (annotated=1 y crudo) usa FileResponse en vez de re-anotar o
# hacer seek OpenCV sobre FUSE en cada request → scrubbing fluido.
os.environ["TJ_WRITE_ANNOTATED"] = "1"
os.environ["TJ_PERSIST_FRAMES"] = "1"
os.environ["TJ_FRAME_CACHE"] = "1"
os.environ["TJ_ANNOTATED_CACHE"] = "1"
os.environ["TJ_FRAME_CACHE_MAX"] = "128"
os.environ["TJ_ANNOTATED_CACHE_MAX"] = "64"

# Cambiar al directorio del engine para que los imports relativos funcionen
os.chdir(str(ENGINE_DIR))
if str(ENGINE_DIR) not in sys.path:
    sys.path.insert(0, str(ENGINE_DIR))

# Symlinks para compatibilidad con paths que usa el engine directamente.
# El clone del repo puede traer dirs reales (venues/, output/) que bloquean unlink().
def safe_symlink(src: pathlib.Path, dst: pathlib.Path):
    """Apunta dst → src. Maneja symlink previo, archivo o directorio real."""
    src = pathlib.Path(src)
    dst = pathlib.Path(dst)
    if not ensure_dir(src):
        raise OSError(
            errno.EOPNOTSUPP,
            f"No existe {src}. Re-ejecuta la celda de Drive "
            "(fallback API) o crea la subcarpeta en Drive web.",
        )

    if dst.is_symlink():
        if dst.resolve() == src.resolve():
            print(f"   ✓ {dst} ya apunta a {src}")
            return
        dst.unlink()
    elif dst.exists():
        if dst.is_dir():
            if any(dst.iterdir()):
                # Preservar contenido local (p.ej. venues/ del repo) y liberar el path
                aside = dst.with_name(dst.name + ".local.bak")
                n = 1
                while aside.exists():
                    aside = dst.with_name(f"{dst.name}.local.bak.{n}")
                    n += 1
                dst.rename(aside)
                print(f"   ⚠ {dst} era un directorio con contenido → movido a {aside.name}")
            else:
                dst.rmdir()
        else:
            dst.unlink()

    dst.symlink_to(src, target_is_directory=True)
    print(f"   🔗 {dst} → {src}")

print("Creando symlinks...")
safe_symlink(DATA_ROOT / "output", ENGINE_DIR / "output")
safe_symlink(DATA_ROOT / "venues", ENGINE_DIR / "venues")

print("✅ Entorno configurado:")
for k in ["HOPLAB_OUTPUT_ROOT", "HOPLAB_VENUE_ROOT", "HOPLAB_VIDEO_ROOT",
          "TJ_WRITE_ANNOTATED", "TJ_PERSIST_FRAMES", "TJ_FRAME_CACHE",
          "TJ_ANNOTATED_CACHE", "TJ_FRAME_CACHE_MAX", "TJ_ANNOTATED_CACHE_MAX",
          "YOLO_CONFIG_DIR"]:
    print(f"   {k} = {os.environ[k]}")


### 2.6 Modelo CNN de pista (venue)

Si `GET /api/venue/model` devuelve `trained:false`, el CNN no está en Drive.

1. En tu PC: `python hoplab-cloud/colab/pack_venue_model.py` (o `.\pack_venue_model.ps1`)
2. Sube `venue-default-weights.zip` a Drive y descomprime en `hoplab-data/venues/default/`  
   (debe quedar `runs/seg/weights/best.pt` + `model.json` + `profile.json`)
3. Ejecuta la celda de abajo para verificar / copiar desde `venues-upload/`
4. Tras arrancar la API, comprueba `trained:true`


In [ ]:
# ─── Verificar / instalar modelo CNN de venue en Drive ───────────────────────
import importlib.util, pathlib

_script = ENGINE_DIR / "hoplab-cloud" / "colab" / "install_venue_model.py"
if not _script.is_file():
    _script = pathlib.Path("hoplab-cloud/colab/install_venue_model.py")

if _script.is_file():
    spec = importlib.util.spec_from_file_location("install_venue_model", _script)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    _status = mod.install(data_root=DATA_ROOT, engine_dir=ENGINE_DIR)
else:
    print(f"⚠️ No encontrado: {_script}")
    _status = {"trained_ready": False}

# Si la API ya corre, comprobar el endpoint
try:
    import urllib.request, json as _json
    with urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/api/venue/model", timeout=5) as r:
        data = _json.loads(r.read().decode())
    print(f"GET /api/venue/model → trained={data.get('trained')}")
except Exception as e:
    print(f"(API aún no disponible para verificar: {e})")
    print("Tras arrancar la API, espera trained:true en /api/venue/model")


## 3. Arrancar API

Arranca uvicorn en un hilo daemon. Los logs van a consola **y** a `DATA_ROOT/logs/hoplab-api.log` (para la Terminal de la sección 5).


In [ ]:
# ─── Arrancar API (con logs a archivo) ───────────────────────────────────────
import logging
import os
import pathlib
import sys
import threading
import time
import urllib.request
import urllib.error
from collections import deque
from datetime import datetime, timezone

# Log path: Drive/logs si hay DATA_ROOT; si no, junto al engine
_log_root = (
    pathlib.Path(DATA_ROOT)
    if "DATA_ROOT" in globals()
    else pathlib.Path("/content/hoplab-data")
)
LOG_DIR = _log_root / "logs"
if not LOG_DIR.is_dir():
    try:
        LOG_DIR.mkdir(parents=True, exist_ok=True)
    except OSError:
        LOG_DIR = pathlib.Path("/content/hoplab-engine/logs")
        LOG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = LOG_DIR / "hoplab-api.log"
os.environ["HOPLAB_API_LOG"] = str(LOG_PATH)

# Ring buffer en memoria (útil si Drive FUSE tarda)
LOG_BUFFER = deque(maxlen=500)


class _TeeStream:
    """Escribe a stream original + archivo + buffer (para el hilo de la API)."""

    def __init__(self, stream, log_file, buffer):
        self._stream = stream
        self._file = log_file
        self._buffer = buffer

    def write(self, data):
        if data:
            try:
                self._stream.write(data)
            except Exception:
                pass
            try:
                self._file.write(data)
                self._file.flush()
            except Exception:
                pass
            for line in data.splitlines():
                if line.strip():
                    self._buffer.append(line)

    def flush(self):
        try:
            self._stream.flush()
        except Exception:
            pass
        try:
            self._file.flush()
        except Exception:
            pass

    def isatty(self):
        return False


def _append_main_log(msg: str) -> None:
    """Escribe desde el hilo principal al archivo + buffer (visible en Terminal)."""
    ts = datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S UTC")
    line = f"[{ts}] {msg}"
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
            f.flush()
    except OSError as e:
        print(f"(no se pudo escribir log: {e})")
    LOG_BUFFER.append(line)


def _setup_api_logging(log_path: pathlib.Path):
    """File + console handlers para uvicorn / fastapi / root (access incluido)."""
    root = logging.getLogger()
    root.setLevel(logging.INFO)
    # Evitar handlers duplicados al re-ejecutar la celda
    for h in list(root.handlers):
        root.removeHandler(h)

    fmt = logging.Formatter("%(asctime)s [%(levelname)s] %(name)s: %(message)s",
                            datefmt="%H:%M:%S")
    fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
    fh.setFormatter(fmt)
    fh.setLevel(logging.INFO)
    sh = logging.StreamHandler(sys.__stdout__)
    sh.setFormatter(fmt)
    sh.setLevel(logging.INFO)
    root.addHandler(fh)
    root.addHandler(sh)

    # FileHandler directo en uvicorn.access (no solo propagate al root)
    for name in ("uvicorn", "uvicorn.error", "uvicorn.access", "fastapi"):
        lg = logging.getLogger(name)
        lg.setLevel(logging.INFO)
        lg.propagate = True
        # Quitar FileHandlers viejos de este logger al re-ejecutar
        for h in list(lg.handlers):
            if isinstance(h, logging.FileHandler):
                lg.removeHandler(h)
                try:
                    h.close()
                except Exception:
                    pass
        access_fh = logging.FileHandler(log_path, mode="a", encoding="utf-8")
        access_fh.setFormatter(fmt)
        access_fh.setLevel(logging.INFO)
        lg.addHandler(access_fh)


def _run_server():
    # Tee stdout/stderr del hilo → archivo + buffer
    log_f = open(LOG_PATH, "a", encoding="utf-8", buffering=1)
    sys.stdout = _TeeStream(sys.__stdout__, log_f, LOG_BUFFER)
    sys.stderr = _TeeStream(sys.__stderr__, log_f, LOG_BUFFER)
    _setup_api_logging(LOG_PATH)

    print(f"[hoplab] API iniciando en :{API_PORT}  log={LOG_PATH}", flush=True)
    import uvicorn
    import api_server  # cwd = ENGINE_DIR
    uvicorn.run(
        api_server.app,
        host="0.0.0.0",
        port=API_PORT,
        log_level="info",
        access_log=True,
    )


# Truncar log de sesión anterior (opcional: comentar para historial largo)
try:
    LOG_PATH.write_text("", encoding="utf-8")
except Exception:
    pass

srv_thread = threading.Thread(target=_run_server, daemon=True, name="hoplab-api")
srv_thread.start()
_append_main_log(f"API thread started (name={srv_thread.name}, port={API_PORT})")
_append_main_log(f"Log path = {LOG_PATH}")

print(f"Esperando API en :{API_PORT} ", end="")
for _ in range(60):
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/status", timeout=2)
        print(" ✅ API lista")
        print(f"📄 Logs → {LOG_PATH}")
        _append_main_log("API ready — /status OK")
        break
    except Exception:
        print(".", end="", flush=True)
        time.sleep(2)
else:
    _append_main_log("API FAILED — no respondió /status en 120s")
    raise RuntimeError(
        "API no respondió en 120s — revisá la Terminal (sección 5) o errores arriba"
    )


## 4. Túnel + URL

URL pública para pegar en Vercel → **Conectar motor**. Esta es la sección que más importa tras el setup.


In [ ]:
# ─── Tunnel cloudflared (gratis, sin cuenta) ─────────────────────────────────
import subprocess, threading, re, time, pathlib

CF_DEB = pathlib.Path("/tmp/cloudflared.deb")

if not pathlib.Path("/usr/bin/cloudflared").exists():
    print("Instalando cloudflared...")
    subprocess.run(
        ["wget", "-q", "-O", str(CF_DEB),
         "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb"],
        check=True)
    subprocess.run(["dpkg", "-i", str(CF_DEB)], check=True, capture_output=True)
    print("Cloudflared instalado.")

# Asegurarse de que no hay config.yaml que bloquee los quick tunnels
cf_cfg = pathlib.Path.home() / ".cloudflared" / "config.yaml"
if cf_cfg.exists():
    cf_cfg.rename(cf_cfg.with_suffix(".yaml.bak"))

TUNNEL_URL = None
tunnel_ready = threading.Event()

def _run_tunnel():
    global TUNNEL_URL
    proc = subprocess.Popen(
        ["cloudflared", "tunnel", "--url", f"http://127.0.0.1:{API_PORT}"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True)
    pattern = re.compile(r"https://[\w-]+\.trycloudflare\.com")
    for line in proc.stdout:
        m = pattern.search(line)
        if m:
            TUNNEL_URL = m.group(0)
            tunnel_ready.set()

t = threading.Thread(target=_run_tunnel, daemon=True)
t.start()

print("Esperando URL del tunnel (máx 60s) ", end="")
ok = tunnel_ready.wait(timeout=60)
if not ok:
    print("\n⚠️  cloudflared no imprimió URL — intenta la celda de fallback debajo")
else:
    print(f"\n✅ Tunnel listo")


In [ ]:
# ─── URL pública + instrucciones ─────────────────────────────────────────────
from IPython.display import display, HTML

if TUNNEL_URL:
    display(HTML(f"""
    <div style="font-family:monospace;background:#1e1e2e;color:#cdd6f4;
                padding:20px;border-radius:12px;font-size:15px;margin:10px 0">
      <b style="font-size:18px;color:#a6e3a1">✅ Motor HopLab listo</b><br><br>

      <b>URL del motor (copiar):</b><br>
      <span style="background:#313244;padding:6px 12px;border-radius:6px;
                   color:#89b4fa;font-size:16px;display:inline-block;margin:8px 0">
        {TUNNEL_URL}
      </span><br><br>

      <b>Siguiente paso:</b><br>
      1. Abre <a href='https://hoplab.vercel.app' style='color:#89dceb'>
            hoplab.vercel.app</a> (o tu URL de Vercel)<br>
      2. Haz clic en <b>Conectar motor</b><br>
      3. Pega la URL de arriba y guarda<br><br>

      <small style='color:#6c7086'>La URL cambia cada sesión de Colab.<br>
      Los análisis se guardan en Drive aunque cierres esta pestaña.<br>
      Dejá la celda <b>Terminal</b> (sección 5) corriendo para ver el feedback del server.</small>
    </div>
    """))
    print(f"\nURL: {TUNNEL_URL}")
    print(f"Docs: {TUNNEL_URL}/docs")
else:
    print("⚠️  Tunnel no disponible. Usa la celda de fallback (localtunnel) abajo.")


### Fallback túnel

Solo si cloudflared no dio URL. Colapsá esta subsección si no la usás.


In [ ]:
# ─── FALLBACK: localtunnel (si cloudflared falla) ────────────────────────────
# Solo ejecutar si cloudflared no imprimió URL (omitir si TUNNEL_URL ya existe).
import subprocess, sys

if globals().get("TUNNEL_URL"):
    print(f"✅ Túnel ya disponible — omitiendo localtunnel.\n   {TUNNEL_URL}")
else:
    subprocess.run(["npm", "install", "-g", "localtunnel", "--silent"],
                   capture_output=True)

    lt_proc = subprocess.Popen(
        ["lt", "--port", str(API_PORT)],
        stdout=subprocess.PIPE, text=True)

    for line in lt_proc.stdout:
        if "loca.lt" in line:
            lt_url = line.strip().split()[-1]
            TUNNEL_URL = lt_url  # para que la Terminal muestre la URL
            print(f"\nFallback URL (localtunnel): {lt_url}")
            print("Nota: localtunnel puede pedir confirmación al abrir en el navegador.")
            print("Pega esta URL en Vercel → Conectar motor.")
            break


## 5. Terminal del server

Feedback en vivo del API (requests, pipeline, errores).

**Ejecutá la celda de código de abajo (play).** Tiene que quedar con ▶ corriendo. Si no ves texto abajo, la celda no se ejecutó.

Dejala corriendo mientras usás el motor. Detener = interrupt (botón Stop).


In [ ]:
# ─── Terminal HopLab (live HTML — Colab-friendly) ────────────────────────────
# Dejá esta celda corriendo (▶). Stop = interrupt. No usa clear_output.
from IPython.display import display, HTML, update_display
import html, time, pathlib, os, urllib.request
from datetime import datetime, timezone

TAIL_LINES = 100
REFRESH = 2.0
HEARTBEAT_EVERY = 10.0  # segundos entre heartbeats al archivo

# 1) Banner inmediato + print (siempre visibles, antes de cualquier sleep)
display(HTML("<h3 style='color:#0a0;font-family:system-ui,sans-serif'>🖥️ Terminal HopLab — iniciando…</h3>"))
print("Terminal activa — si ves este mensaje, la celda corre.")

# 2) Puerto obligatorio (celda config / API primero)
if "API_PORT" not in globals() or not API_PORT:
    display(HTML(
        "<div style='color:#fff;background:#b91c1c;padding:14px;border-radius:8px;font-family:system-ui,sans-serif'>"
        "<b>Error:</b> falta <code>API_PORT</code>. Ejecutá primero la celda de "
        "<b>configuración (0)</b> y <b>Arrancar API (3)</b>, después re-ejecutá esta Terminal."
        "</div>"
    ))
    raise RuntimeError("API_PORT no definido — ejecutá secciones 0 y 3 antes")

# 3) Resolver LOG_PATH
_log_candidates = []
if os.environ.get("HOPLAB_API_LOG"):
    _log_candidates.append(pathlib.Path(os.environ["HOPLAB_API_LOG"]))
if "LOG_PATH" in globals() and LOG_PATH:
    _log_candidates.append(pathlib.Path(str(LOG_PATH)))
if "DATA_ROOT" in globals() and DATA_ROOT:
    _log_candidates.append(pathlib.Path(DATA_ROOT) / "logs" / "hoplab-api.log")
_log_candidates.append(pathlib.Path("/content/hoplab-api.log"))

LOG_PATH = None
_log_note = ""
for _cand in _log_candidates:
    try:
        _cand.parent.mkdir(parents=True, exist_ok=True)
        if not _cand.exists():
            _cand.touch()
        LOG_PATH = _cand
        break
    except OSError:
        continue

if LOG_PATH is None:
    LOG_PATH = pathlib.Path("/content/hoplab-api.log")
    try:
        LOG_PATH.touch()
    except OSError:
        pass
    _log_note = " (fallback local)"
elif str(LOG_PATH) == "/content/hoplab-api.log" and not os.environ.get("HOPLAB_API_LOG"):
    _log_note = " (creado local — API aún no escribió HOPLAB_API_LOG)"

os.environ.setdefault("HOPLAB_API_LOG", str(LOG_PATH))
print(f"Log: {LOG_PATH}{_log_note}")


def _api_status() -> str:
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{API_PORT}/status", timeout=1.5)
        return "🟢 API OK"
    except Exception:
        return "🔴 API no responde"


def _thread_alive() -> str:
    thr = globals().get("srv_thread")
    if thr is None:
        return "hilo API: (no iniciado — ejecutá sección 3)"
    try:
        return f"hilo API: {'vivo' if thr.is_alive() else 'MUERTO'}"
    except Exception as e:
        return f"hilo API: error ({e})"


def _is_status_noise(line: str) -> bool:
    """Filtra health-checks GET /status (Terminal + probes) del tail visible."""
    s = (line or "").lower()
    if "/status" not in s:
        return False
    # access-log tipico: GET /status 200
    if "get /status" in s:
        return True
    return False


def _read_tail(path: pathlib.Path, n: int) -> list:
    lines = []
    warn = []
    exists = path.exists() if path else False
    empty = True
    status_filtered = 0
    if exists:
        try:
            with open(path, "r", encoding="utf-8", errors="replace") as f:
                raw = f.readlines()
            # Leer mas lineas crudas para compensar filtrado de /status
            raw_tail = [ln.rstrip("\n") for ln in raw[-(n * 4):]]
            kept = []
            for ln in raw_tail:
                if _is_status_noise(ln):
                    status_filtered += 1
                else:
                    kept.append(ln)
            lines = kept[-n:]
            empty = not any(ln.strip() for ln in lines)
        except OSError as e:
            warn.append(f"(error leyendo log: {e})")
            empty = True
    else:
        warn.append(f"(archivo aun no existe: {path})")

    if empty or not exists:
        buf = globals().get("LOG_BUFFER")
        if buf:
            kept = []
            for ln in list(buf)[-(n * 4):]:
                if _is_status_noise(ln):
                    status_filtered += 1
                else:
                    kept.append(ln)
            lines = kept[-n:]
            warn.append("(mostrando LOG_BUFFER en memoria)")
        elif not lines:
            warn.append("(log vacio — esperando salida del API o heartbeat)")

    if status_filtered:
        warn.append(f"(ocultas {status_filtered} lineas GET /status)")
    warn.append(_thread_alive())
    return warn + lines if warn else lines

def _append_heartbeat(path: pathlib.Path) -> None:
    ts = datetime.now(timezone.utc).strftime("%H:%M:%S")
    msg = f"[{ts}] [terminal] watching… {_api_status()} · {_thread_alive()}\n"
    try:
        with open(path, "a", encoding="utf-8") as f:
            f.write(msg)
    except OSError:
        pass


handle = display(
    HTML("<pre style='font-family:monospace'>esperando logs…</pre>"),
    display_id=True,
)
_last_hb = 0.0

try:
    while True:
        now = time.time()
        if now - _last_hb >= HEARTBEAT_EVERY:
            _append_heartbeat(LOG_PATH)
            _last_hb = now

        url = globals().get("TUNNEL_URL") or "(túnel no listo — ejecutá sección 4)"
        # No spamear GET /status cada refresco: cachear ~10s
        if now - getattr(_read_tail, "_status_ts", 0) >= 10.0:
            _read_tail._status_cache = _api_status()
            _read_tail._status_ts = now
        status = getattr(_read_tail, "_status_cache", None) or _api_status()
        lines = _read_tail(LOG_PATH, TAIL_LINES)
        body = html.escape("\n".join(lines) if lines else "(sin líneas)")

        # Advertencia visual si el log sigue vacío / hilo muerto
        extra = ""
        thr = globals().get("srv_thread")
        log_empty = (not LOG_PATH.exists()) or (LOG_PATH.stat().st_size == 0 if LOG_PATH.exists() else True)
        if thr is not None and not thr.is_alive():
            extra = (
                "<div style='color:#f85149;margin-bottom:8px'>"
                "⚠️ El hilo de la API no está vivo. Re-ejecutá la sección 3 (Arrancar API)."
                "</div>"
            )
        elif log_empty and not globals().get("LOG_BUFFER"):
            extra = (
                "<div style='color:#d29922;margin-bottom:8px'>"
                "⚠️ Log vacío por ahora. Si la API arrancó, deberías ver heartbeats "
                "[terminal] watching… cada ~10s. Si no, revisá sección 3."
                "</div>"
            )

        handle.update(HTML(f"""
        <div style="font-family:monospace;border:2px solid #333;padding:12px;background:#111;color:#d4d4d4;border-radius:8px">
          <div style="color:#7ee787;margin-bottom:8px">HOPLAB TERMINAL · {status} · :{API_PORT}</div>
          <div style="color:#79c0ff;margin-bottom:8px">URL: {html.escape(str(url))}</div>
          <div style="color:#aaa;margin-bottom:8px">Log: {html.escape(str(LOG_PATH))} · refresco {REFRESH}s · Stop para detener</div>
          {extra}
          <pre style="white-space:pre-wrap;max-height:420px;overflow:auto;margin:0">{body}</pre>
        </div>
        """))
        time.sleep(REFRESH)
except KeyboardInterrupt:
    url = globals().get("TUNNEL_URL") or "(túnel no listo)"
    tail = html.escape("\n".join(_read_tail(LOG_PATH, min(40, TAIL_LINES))))
    handle.update(HTML(f"""
    <div style="font-family:monospace;border:2px solid #555;padding:12px;background:#1a1a1a;color:#d4d4d4;border-radius:8px">
      <div style="color:#f85149;margin-bottom:8px">Terminal detenida (interrupt).</div>
      <div style="color:#79c0ff;margin-bottom:8px">URL: {html.escape(str(url))}</div>
      <div style="color:#aaa;margin-bottom:8px">Re-ejecutá esta celda para volver a mirar el log.</div>
      <pre style="white-space:pre-wrap;max-height:320px;overflow:auto;margin:0">{tail}</pre>
    </div>
    """))
    print("Terminal detenida. Re-ejecutá la celda para reanudar.")
